# 06. REPRODUCIBILIDAD Y DESPLIEGUE DEL SISTEMA PREDICTIVO

La utilidad de un sistema predictivo no depende únicamente de su rendimiento durante la evaluación, sino también de la posibilidad de reconstruir posteriormente su procedimiento de inferencia. Para ello, deben conservarse de forma independiente los componentes responsables del preprocesamiento, el modelo entrenado y la configuración necesaria para transformar nuevas observaciones y generar predicciones sin depender del estado en memoria de los notebooks utilizados durante el desarrollo.

El sistema definitivo procede del protocolo temporal establecido durante el modelado. La selección del modelo, sus hiperparámetros y el umbral de decisión se realizó utilizando exclusivamente información disponible hasta 2025, manteniendo enero–mayo de 2026 como período externo independiente. Tras finalizar este proceso quedaron fijados el codificador de variables categóricas, el escalador de variables numéricas, el modelo de regresión logística entrenado sobre 2022–2025 y el umbral de decisión de 0.406588.

Este notebook se centra exclusivamente en la reproducibilidad del procedimiento resultante. No se realizan nuevos procesos de entrenamiento, selección de variables, optimización de hiperparámetros o modificación del umbral. En su lugar, los artefactos definitivos se recuperan directamente desde disco y se utilizan para reconstruir el mismo flujo de transformación e inferencia empleado durante la evaluación final.

La reproducibilidad se comprobará mediante la recuperación de los componentes persistidos, la reconstrucción del esquema de entrada y de las transformaciones aplicadas por el modelo, y la comparación de nuevas inferencias con resultados previamente almacenados. Finalmente, se verificará la capacidad del procedimiento reconstruido para procesar nuevas observaciones compatibles y se documentará el conjunto mínimo de artefactos necesario para su utilización posterior.

El término despliegue se emplea en este contexto como preparación técnica del sistema para ejecutar inferencias reproducibles fuera del proceso de entrenamiento. Por tanto, no implica la construcción de una infraestructura productiva en tiempo real, sino la separación entre el desarrollo experimental del modelo y su posterior utilización sobre nuevos datos.

El notebook se organizará en los siguientes bloques:

1. Configuración del entorno y recuperación de artefactos.
2. Reconstrucción del procedimiento de inferencia.
3. Validación de la reproducibilidad predictiva.
4. Ejecución sobre nuevas observaciones.
5. Persistencia de la configuración reproducible.

## 1. Configuración del entorno y recuperación de artefactos

La primera etapa consiste en reconstruir el entorno mínimo necesario para utilizar el sistema predictivo a partir de los productos persistidos durante el modelado.

A diferencia de los notebooks anteriores, no se recuperarán datasets completos ni resultados destinados exclusivamente al análisis exploratorio. El objetivo es identificar únicamente los componentes necesarios para ejecutar inferencia: configuración del modelo, codificador de variables categóricas, escalador de variables numéricas, modelo entrenado y umbral definitivo de decisión.

La recuperación desde disco es metodológicamente relevante porque permite comprobar que el funcionamiento del sistema no depende de objetos conservados en memoria durante el proceso de entrenamiento. Cada componente deberá proceder de los artefactos definitivos generados previamente, sin reconstrucciones manuales que puedan introducir diferencias respecto al modelo evaluado.

En este bloque se abordarán progresivamente:

1. Configuración de rutas y entorno de ejecución.
2. Identificación de los artefactos persistidos.
3. Recuperación de los componentes definitivos.
4. Validación consolidada de la configuración recuperada.

El resultado esperado será disponer en memoria de todos los componentes necesarios para reconstruir posteriormente el procedimiento de inferencia, manteniendo intacta la configuración utilizada por el modelo definitivo.

### 1.1 Configuración de rutas y entorno de ejecución

El procedimiento de reproducibilidad comienza estableciendo las dependencias y rutas necesarias para recuperar los componentes persistidos del sistema predictivo. La configuración se limita a los recursos requeridos para la inferencia, evitando incorporar productos analíticos que no intervienen en la generación de nuevas predicciones.

El directorio `results/modeling` constituye la fuente principal de artefactos del modelo definitivo. Su contenido se inspeccionará en el siguiente subbloque para identificar los archivos disponibles antes de proceder a su recuperación.

Esta separación permite que el notebook pueda ejecutarse desde una sesión independiente y evita asumir la existencia en memoria de objetos generados durante el entrenamiento.

In [1]:
# ---------------------------------------------------------
# 1. Importación de dependencias
# ---------------------------------------------------------

from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 2. Configuración de rutas del proyecto
# ---------------------------------------------------------

project_root = Path(r"G:\My Drive\MASTER Big Data\TFM")

data_path = project_root / "data"
results_path = project_root / "results"
modeling_path = results_path / "modeling"


# ---------------------------------------------------------
# 3. Validación básica de las rutas necesarias
# ---------------------------------------------------------

paths_status = pd.DataFrame(
    {
        "resource": [
            "project_root",
            "data_path",
            "results_path",
            "modeling_path",
        ],
        "path": [
            str(project_root),
            str(data_path),
            str(results_path),
            str(modeling_path),
        ],
        "exists": [
            project_root.exists(),
            data_path.exists(),
            results_path.exists(),
            modeling_path.exists(),
        ],
    }
)

display(paths_status)

,resource,path,exists
0,project_root,G:\My Drive\MASTER Big Data\TFM,True
1,data_path,G:\My Drive\MASTER Big Data\TFM\data,True
2,results_path,G:\My Drive\MASTER Big Data\TFM\results,True
3,modeling_path,G:\My Drive\MASTER Big Data\TFM\results\modeling,True


### 1.2 Identificación de los artefactos persistidos

Una vez confirmada la disponibilidad del directorio de modelado, se inspeccionarán los archivos persistidos durante el entrenamiento y evaluación del sistema.

Esta etapa tiene como finalidad identificar los artefactos realmente disponibles antes de proceder a su recuperación. De este modo, la reconstrucción del sistema se basará exclusivamente en productos existentes y no en nombres de archivos o configuraciones asumidas manualmente.

Para cada archivo se registrará su nombre, extensión y tamaño, información suficiente para reconocer los componentes persistidos y determinar cuáles intervienen directamente en el procedimiento de inferencia.

In [2]:
# ---------------------------------------------------------
# 1. Recuperación dinámica de los archivos persistidos
# ---------------------------------------------------------

modeling_files = sorted(
    file
    for file in modeling_path.rglob("*")
    if file.is_file()
)


# ---------------------------------------------------------
# 2. Construcción del inventario
# ---------------------------------------------------------

artifacts_inventory = pd.DataFrame(
    [
        {
            "file": file.name,
            "extension": file.suffix.lower(),
            "relative_path": str(file.relative_to(modeling_path)),
            "size_mb": file.stat().st_size / (1024 ** 2),
        }
        for file in modeling_files
    ]
)

artifacts_inventory["size_mb"] = artifacts_inventory["size_mb"].round(4)


# ---------------------------------------------------------
# 3. Resumen de los artefactos disponibles
# ---------------------------------------------------------

print(f"Archivos identificados: {len(artifacts_inventory):,}")
print(f"Tamaño total: {artifacts_inventory['size_mb'].sum():,.2f} MB")

display(artifacts_inventory)

Archivos identificados: 13
Tamaño total: 29.02 MB


,file,extension,relative_path,size_mb
0,external_test_results_2026.csv,.csv,external_test_results_2026.csv,0.0004
1,final_categorical_encoder_2022_2025.joblib,.joblib,final_categorical_encoder_2022_2025.joblib,0.0168
2,final_logistic_regression_2022_2025.joblib,.joblib,final_logistic_regression_2022_2025.joblib,0.0041
3,final_model_configuration.json,.json,final_model_configuration.json,0.0007
4,final_numerical_scaler_2022_2025.joblib,.joblib,final_numerical_scaler_2022_2025.joblib,0.0010
5,finalist_temporal_validation_results.csv,.csv,finalist_temporal_validation_results.csv,0.0010
6,grid_search_logistic_screening.csv,.csv,grid_search_logistic_screening.csv,0.0023
7,grid_search_random_forest_screening.csv,.csv,grid_search_random_forest_screening.csv,0.0033
8,imbalance_results.csv,.csv,imbalance_results.csv,0.0020
9,initial_models_results.csv,.csv,initial_models_results.csv,0.0005


### 1.3 Recuperación de los componentes definitivos

Una vez identificados los artefactos disponibles, se recuperarán exclusivamente los componentes necesarios para reconstruir el sistema de inferencia: el codificador de variables categóricas, el escalador de variables numéricas, el modelo final de regresión logística y su configuración persistida.

Los tres objetos de procesamiento y modelado se cargarán directamente desde los archivos `.joblib` generados durante el entrenamiento definitivo sobre 2022–2025. La configuración se recuperará desde `final_model_configuration.json`, evitando redefinir manualmente variables, parámetros o el umbral de decisión.

Este procedimiento garantiza que las etapas posteriores utilicen exactamente los componentes persistidos del sistema evaluado, sin introducir una nueva estimación o modificación del modelo.

In [3]:
# ---------------------------------------------------------
# 1. Importación adicional para recuperar la configuración
# ---------------------------------------------------------

import json


# ---------------------------------------------------------
# 2. Definición de los artefactos definitivos
# ---------------------------------------------------------

encoder_path = modeling_path / "final_categorical_encoder_2022_2025.joblib"
scaler_path = modeling_path / "final_numerical_scaler_2022_2025.joblib"
model_path = modeling_path / "final_logistic_regression_2022_2025.joblib"
configuration_path = modeling_path / "final_model_configuration.json"


# ---------------------------------------------------------
# 3. Recuperación de los componentes persistidos
# ---------------------------------------------------------

categorical_encoder = joblib.load(encoder_path)
numerical_scaler = joblib.load(scaler_path)
final_model = joblib.load(model_path)

with open(configuration_path, "r", encoding="utf-8") as file:
    final_model_configuration = json.load(file)


# ---------------------------------------------------------
# 4. Resumen de los componentes recuperados
# ---------------------------------------------------------

print("Componentes recuperados:")
print(f"- Codificador categórico: {type(categorical_encoder).__name__}")
print(f"- Escalador numérico: {type(numerical_scaler).__name__}")
print(f"- Modelo predictivo: {type(final_model).__name__}")
print(f"- Configuración: {type(final_model_configuration).__name__}")

print("\nConfiguración persistida:")
display(pd.DataFrame(
    {
        "parameter": list(final_model_configuration.keys()),
        "value": [
            str(value)
            for value in final_model_configuration.values()
        ],
    }
))

Componentes recuperados:
- Codificador categórico: OneHotEncoder
- Escalador numérico: StandardScaler
- Modelo predictivo: LogisticRegression
- Configuración: dict

Configuración persistida:


,parameter,value
0,model,Logistic Regression
1,training_period,2022-2025
2,external_test_period,2026-01 to 2026-05
3,C,0.1
4,tol,0.01
5,solver,saga
6,penalty,l2
7,class_weight,balanced
8,max_iter,500
9,random_state,42


### 1.4 Validación consolidada de la configuración recuperada

La recuperación correcta de los archivos no garantiza por sí sola que los componentes persistidos sean estructuralmente compatibles. Antes de reconstruir el procedimiento de inferencia, se realizará una validación consolidada entre la configuración, el codificador categórico, el escalador numérico y el modelo final.

La comprobación se centrará en las características necesarias para aplicar el sistema: número de variables categóricas y numéricas, dimensionalidad resultante del preprocesamiento y número de características esperado por la regresión logística.

También se verificará que los principales parámetros almacenados en la configuración coincidan con los parámetros efectivos del modelo recuperado. Esta comprobación permite detectar posibles inconsistencias entre artefactos sin repetir las evaluaciones predictivas realizadas en notebooks anteriores.

Si todas las condiciones son compatibles, el bloque quedará validado y los componentes podrán utilizarse directamente para reconstruir el procedimiento de inferencia.

In [4]:
# ---------------------------------------------------------
# 1. Recuperación de la estructura esperada
# ---------------------------------------------------------

categorical_features = final_model_configuration["categorical_features"]
numerical_features = final_model_configuration["numerical_features"]
expected_transformed_features = final_model_configuration["transformed_features"]
decision_threshold = final_model_configuration["decision_threshold"]

encoder_input_features = categorical_encoder.n_features_in_
scaler_input_features = numerical_scaler.n_features_in_
model_input_features = final_model.n_features_in_

encoder_output_features = len(
    categorical_encoder.get_feature_names_out(categorical_features)
)

total_transformed_features = (
    encoder_output_features + scaler_input_features
)


# ---------------------------------------------------------
# 2. Comprobación consolidada de compatibilidad
# ---------------------------------------------------------

validation_checks = pd.DataFrame(
    [
        {
            "check": "Variables categóricas",
            "expected": len(categorical_features),
            "observed": encoder_input_features,
            "valid": len(categorical_features) == encoder_input_features,
        },
        {
            "check": "Variables numéricas",
            "expected": len(numerical_features),
            "observed": scaler_input_features,
            "valid": len(numerical_features) == scaler_input_features,
        },
        {
            "check": "Dimensión transformada",
            "expected": expected_transformed_features,
            "observed": total_transformed_features,
            "valid": expected_transformed_features == total_transformed_features,
        },
        {
            "check": "Entrada del modelo",
            "expected": expected_transformed_features,
            "observed": model_input_features,
            "valid": expected_transformed_features == model_input_features,
        },
        {
            "check": "Parámetro C",
            "expected": final_model_configuration["C"],
            "observed": final_model.C,
            "valid": final_model_configuration["C"] == final_model.C,
        },
        {
            "check": "Solver",
            "expected": final_model_configuration["solver"],
            "observed": final_model.solver,
            "valid": final_model_configuration["solver"] == final_model.solver,
        },
        {
            "check": "Class weight",
            "expected": final_model_configuration["class_weight"],
            "observed": final_model.class_weight,
            "valid": final_model_configuration["class_weight"] == final_model.class_weight,
        },
    ]
)

configuration_valid = validation_checks["valid"].all()


# ---------------------------------------------------------
# 3. Resultado consolidado
# ---------------------------------------------------------

display(validation_checks)

print(f"\nUmbral recuperado: {decision_threshold:.6f}")
print(
    "Configuración reproducible: "
    f"{'SÍ' if configuration_valid else 'NO'}"
)

,check,expected,observed,valid
0,Variables categóricas,8,8,True
1,Variables numéricas,2,2,True
2,Dimensión transformada,853,853,True
3,Entrada del modelo,853,853,True
4,Parámetro C,0.1,0.1,True
5,Solver,saga,saga,True
6,Class weight,balanced,balanced,True



Umbral recuperado: 0.406588
Configuración reproducible: SÍ


## 2. Reconstrucción del procedimiento de inferencia

Una vez recuperados y validados los artefactos definitivos, se reconstruirá el procedimiento utilizado para transformar nuevas observaciones en resultados predictivos.

El sistema recibe 10 variables de entrada: 8 categóricas y 2 numéricas. Las variables categóricas deben procesarse mediante el `OneHotEncoder` persistido, mientras que las variables numéricas deben transformarse mediante el `StandardScaler` utilizado durante el entrenamiento. Ambas representaciones se combinan posteriormente respetando el mismo orden empleado durante la construcción del modelo.

El resultado del preprocesamiento debe contener 853 características, que constituyen la entrada esperada por la regresión logística definitiva. A partir de esta representación, el modelo genera un `risk_score` asociado a la clase positiva `ARR_DEL15 = 1`.

La decisión operativa no utilizará el umbral convencional de 0.5, sino el umbral de 0.406588 seleccionado previamente sobre la validación temporal de 2025. Una observación será clasificada como alerta cuando su `risk_score` sea igual o superior a dicho valor.

El procedimiento puede representarse mediante la siguiente secuencia:

`variables de entrada → transformación categórica y numérica → 853 características → regresión logística → risk_score → umbral 0.406588 → alert`

La reconstrucción mantendrá separados los conceptos de puntuación y decisión. `risk_score` representa la puntuación generada por el modelo, mientras que `alert` constituye la decisión binaria obtenida al aplicar el umbral operativo. La puntuación no se interpretará como una probabilidad calibrada de retraso.

En este bloque se abordarán progresivamente:

1. Definición del esquema de entrada.
2. Reconstrucción de las transformaciones de preprocesamiento.
3. Reconstrucción de la generación de puntuaciones y alertas.
4. Integración del procedimiento completo de inferencia.

El resultado esperado será disponer de un procedimiento reproducible que transforme observaciones compatibles con el esquema original en `risk_score` y `alert`, utilizando exclusivamente los artefactos definitivos recuperados desde disco.

### 2.1 Definición del esquema de entrada

La ejecución del sistema requiere que cada nueva observación proporcione las mismas variables utilizadas durante el entrenamiento y que estas se encuentren organizadas en el mismo esquema lógico.

La configuración persistida permite recuperar esta estructura sin definir manualmente las características. El esquema está formado por 8 variables categóricas —mes, día de la semana, aerolínea comercializadora, aerolínea operadora, aeropuerto de origen, aeropuerto de destino y franjas horarias previstas de salida y llegada— y 2 variables numéricas correspondientes a duración prevista y distancia.

`FL_DATE` puede conservarse como identificador temporal de la observación, pero no forma parte de las características transformadas que recibe el modelo. De forma equivalente, `ARR_DEL15` únicamente estará disponible cuando exista un resultado observado y se requiera evaluar posteriormente la predicción; no constituye una variable de entrada.

Esta distinción permite definir explícitamente qué información necesita el sistema para generar una predicción y evita incorporar accidentalmente variables posteriores al vuelo.

In [5]:
# ---------------------------------------------------------
# 1. Recuperación del esquema predictivo desde la configuración
# ---------------------------------------------------------

input_features = categorical_features + numerical_features

input_schema = pd.DataFrame(
    {
        "feature": input_features,
        "feature_type": (
            ["Categórica"] * len(categorical_features)
            + ["Numérica"] * len(numerical_features)
        ),
        "model_input": True,
    }
)


# ---------------------------------------------------------
# 2. Resumen del esquema requerido
# ---------------------------------------------------------

print(f"Variables de entrada requeridas: {len(input_features)}")
print(f"- Categóricas: {len(categorical_features)}")
print(f"- Numéricas: {len(numerical_features)}")

display(input_schema)

Variables de entrada requeridas: 10
- Categóricas: 8
- Numéricas: 2


,feature,feature_type,model_input
0,MONTH,Categórica,True
1,DAY_OF_WEEK,Categórica,True
2,MKT_UNIQUE_CARRIER,Categórica,True
3,OP_UNIQUE_CARRIER,Categórica,True
4,ORIGIN,Categórica,True
5,DEST,Categórica,True
6,DEP_TIME_BLK,Categórica,True
7,ARR_TIME_BLK,Categórica,True
8,CRS_ELAPSED_TIME,Numérica,True
9,DISTANCE,Numérica,True


### 2.2 Reconstrucción de las transformaciones de preprocesamiento

El siguiente paso consiste en reproducir la transformación que convierte las 10 variables originales en la representación numérica utilizada por la regresión logística.

Las variables categóricas se procesarán mediante el `OneHotEncoder` recuperado, mientras que las variables numéricas se transformarán mediante el `StandardScaler` persistido. Posteriormente, ambas salidas se combinarán respetando el mismo orden utilizado durante el entrenamiento.

Para comprobar inicialmente este procedimiento se utilizará una observación de prueba construida a partir de categorías conocidas por el codificador. Esta observación no se utilizará para evaluar el rendimiento del modelo, sino exclusivamente para verificar que el procedimiento de transformación puede ejecutarse de forma independiente.

El resultado esperado para cada observación es una representación de 853 características, coincidente con la dimensionalidad de entrada de la regresión logística definitiva.

In [7]:
# ---------------------------------------------------------
# 1. Construcción de una observación compatible
# ---------------------------------------------------------

sample_data = {
    feature: [categorical_encoder.categories_[index][0]]
    for index, feature in enumerate(categorical_features)
}

for index, feature in enumerate(numerical_features):
    sample_data[feature] = [float(numerical_scaler.mean_[index])]

sample_input = pd.DataFrame(
    sample_data,
    columns=input_features,
)


# ---------------------------------------------------------
# 2. Aplicación de los transformadores persistidos
# ---------------------------------------------------------

sample_categorical = categorical_encoder.transform(
    sample_input[categorical_features]
)

sample_numerical = numerical_scaler.transform(
    sample_input[numerical_features]
)


# ---------------------------------------------------------
# 3. Combinación de las características transformadas
# ---------------------------------------------------------

from scipy.sparse import csr_matrix, hstack

sample_numerical_sparse = csr_matrix(sample_numerical)

sample_transformed = hstack(
    [sample_categorical, sample_numerical_sparse],
    format="csr",
)


# ---------------------------------------------------------
# 4. Resumen de la transformación
# ---------------------------------------------------------

print(f"Observaciones de entrada: {sample_input.shape[0]:,}")
print(f"Variables originales: {sample_input.shape[1]:,}")
print(f"Características categóricas transformadas: {sample_categorical.shape[1]:,}")
print(f"Características numéricas transformadas: {sample_numerical.shape[1]:,}")
print(f"Dimensión final: {sample_transformed.shape}")

display(sample_input)

Observaciones de entrada: 1
Variables originales: 10
Características categóricas transformadas: 851
Características numéricas transformadas: 2
Dimensión final: (1, 853)


,MONTH,DAY_OF_WEEK,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,ORIGIN,DEST,DEP_TIME_BLK,ARR_TIME_BLK,CRS_ELAPSED_TIME,DISTANCE
0,1,1,AA,9E,ABE,ABE,0001-0559,0001-0559,143.106871,804.291599


### 2.3 Reconstrucción de la generación de puntuaciones y alertas

Una vez obtenida la representación transformada de 853 características, esta puede utilizarse directamente como entrada de la regresión logística persistida.

El modelo genera mediante `predict_proba` una puntuación asociada a cada una de sus clases. Para el sistema de alerta se recuperará específicamente la puntuación correspondiente a la clase positiva `ARR_DEL15 = 1`, evitando asumir manualmente su posición dentro de la salida del modelo.

La decisión binaria se obtendrá posteriormente comparando `risk_score` con el umbral definitivo de 0.406588. Cuando la puntuación sea igual o superior al umbral se generará `alert = 1`; en caso contrario, `alert = 0`.

Esta separación mantiene diferenciadas la salida continua del modelo y la decisión operativa. El `risk_score` se conservará como puntuación de riesgo y no se interpretará como una probabilidad calibrada de retraso.

In [8]:
# ---------------------------------------------------------
# 1. Identificación de la clase positiva
# ---------------------------------------------------------

positive_class = 1

positive_class_index = np.where(
    final_model.classes_ == positive_class
)[0][0]


# ---------------------------------------------------------
# 2. Generación de la puntuación de riesgo
# ---------------------------------------------------------

sample_risk_score = final_model.predict_proba(
    sample_transformed
)[:, positive_class_index]


# ---------------------------------------------------------
# 3. Aplicación del umbral definitivo
# ---------------------------------------------------------

sample_alert = (
    sample_risk_score >= decision_threshold
).astype("int8")


# ---------------------------------------------------------
# 4. Construcción del resultado de inferencia
# ---------------------------------------------------------

sample_prediction = pd.DataFrame(
    {
        "risk_score": sample_risk_score,
        "decision_threshold": decision_threshold,
        "alert": sample_alert,
    }
)

display(sample_prediction)

print(f"Clase positiva: ARR_DEL15 = {positive_class}")
print(f"Umbral aplicado: {decision_threshold:.6f}")

,risk_score,decision_threshold,alert
0,0.276551,0.406588,0


Clase positiva: ARR_DEL15 = 1
Umbral aplicado: 0.406588


#### Interpretación

El modelo generó correctamente un `risk_score` de 0.276551 para la observación de prueba. Al encontrarse por debajo del umbral definitivo de 0.406588, la observación fue clasificada como `alert = 0`.

El resultado confirma que los artefactos recuperados permiten completar la secuencia desde las características transformadas hasta la decisión operativa, manteniendo separados el `risk_score` y la alerta binaria.

### 2.4 Integración del procedimiento completo de inferencia

Una vez reconstruidas individualmente las etapas de preprocesamiento, generación de puntuaciones y aplicación del umbral, se integrarán en un único procedimiento de inferencia.

El procedimiento recibirá un `DataFrame` con las 10 variables requeridas por el modelo, comprobará la disponibilidad de dichas características, aplicará los transformadores persistidos y generará finalmente `risk_score` y `alert`.

La función no realizará ningún ajuste sobre los datos recibidos. Tanto el `OneHotEncoder` como el `StandardScaler` y la regresión logística permanecerán en el estado recuperado desde disco, garantizando que la ejecución de nuevas observaciones no modifique los componentes definitivos del sistema.

La salida conservará las variables originales y añadirá exclusivamente las dos variables generadas por el sistema predictivo. Esta estructura facilita posteriormente identificar cada observación junto con su puntuación de riesgo y decisión de alerta.

In [15]:
# ---------------------------------------------------------
# 1. Definición del procedimiento completo de inferencia
# ---------------------------------------------------------

def predict_alerts(input_data):
    """Genera puntuaciones de riesgo y alertas con los artefactos persistidos."""

    # Comprobar que estén disponibles todas las variables requeridas
    missing_features = [
        feature
        for feature in input_features
        if feature not in input_data.columns
    ]

    if missing_features:
        raise ValueError(
            f"Faltan variables requeridas: {missing_features}"
        )

    # Seleccionar y ordenar las variables según la configuración definitiva
    model_input = input_data[input_features].copy()

    # Aplicar el preprocesamiento persistido
    categorical_transformed = categorical_encoder.transform(
        model_input[categorical_features]
    )

    numerical_transformed = numerical_scaler.transform(
        model_input[numerical_features]
    )

    numerical_transformed = csr_matrix(
        numerical_transformed
    )

    transformed_input = hstack(
        [
            categorical_transformed,
            numerical_transformed,
        ],
        format="csr",
    )

    # Generar la puntuación correspondiente a ARR_DEL15 = 1
    risk_score = final_model.predict_proba(
        transformed_input
    )[:, positive_class_index]

    # Aplicar el umbral operativo definitivo
    alert = (
        risk_score >= decision_threshold
    ).astype("int8")

    # Incorporar los resultados a las observaciones originales
    prediction_output = input_data.copy()

    prediction_output["risk_score"] = risk_score
    prediction_output["alert"] = alert

    return prediction_output


# ---------------------------------------------------------
# 2. Comprobación del procedimiento integrado
# ---------------------------------------------------------

sample_prediction_integrated = predict_alerts(
    sample_input
)

display(sample_prediction_integrated)

print(
    "Coincidencia del risk_score: "
    f"{np.allclose(sample_prediction_integrated['risk_score'], sample_risk_score)}"
)

print(
    "Coincidencia de la alerta: "
    f"{np.array_equal(sample_prediction_integrated['alert'].to_numpy(), sample_alert)}"
)


,MONTH,DAY_OF_WEEK,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,ORIGIN,DEST,DEP_TIME_BLK,ARR_TIME_BLK,CRS_ELAPSED_TIME,DISTANCE,risk_score,alert
0,1,1,AA,9E,ABE,ABE,0001-0559,0001-0559,143.106871,804.291599,0.276551,0


Coincidencia del risk_score: True
Coincidencia de la alerta: True


#### Interpretación

El procedimiento integrado reproduce exactamente los resultados obtenidos mediante las etapas ejecutadas por separado. Tanto el `risk_score` de 0.276551 como la decisión `alert = 0` coinciden, confirmando que la secuencia completa de inferencia ha quedado correctamente encapsulada.

A partir de este punto, nuevas observaciones compatibles con las 10 variables de entrada pueden procesarse directamente mediante `predict_alerts()` sin reajustar el preprocesamiento, reentrenar el modelo ni redefinir el umbral.

## 3. Validación de la reproducibilidad predictiva

La reproducibilidad del sistema se comprobará mediante observaciones reales del período externo de 2026 cuyos resultados predictivos ya fueron persistidos previamente.

La finalidad no es volver a evaluar el modelo, sino verificar que la recuperación independiente del codificador, escalador, modelo y umbral permite reproducir las mismas salidas a partir de las variables originales.

Para ello se seleccionará una muestra controlada de observaciones, se ejecutará nuevamente el procedimiento `predict_alerts()` y se compararán los nuevos `risk_score` y `alert` con los valores previamente almacenados.

El criterio principal será la coincidencia de las alertas y la equivalencia numérica de las puntuaciones de riesgo.


In [16]:
# ---------------------------------------------------------
# 1. Recuperación de una muestra real previamente predicha
# ---------------------------------------------------------

import pyarrow.parquet as pq

prediction_source_path = (
    results_path
    / "operational_analysis"
    / "external_alert_analysis_2026.parquet"
)

sample_size = 10_000

prediction_parquet = pq.ParquetFile(prediction_source_path)

sample_batch = next(
    prediction_parquet.iter_batches(
        batch_size=sample_size,
        columns=input_features + ["risk_score", "alert"],
    )
)

reproducibility_sample = (
    sample_batch
    .to_pandas()
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 2. Reejecución de la inferencia
# ---------------------------------------------------------

new_predictions = predict_alerts(
    reproducibility_sample[input_features]
)


# ---------------------------------------------------------
# 3. Comparación con los resultados persistidos
# ---------------------------------------------------------

risk_score_difference = np.abs(
    new_predictions["risk_score"].to_numpy()
    - reproducibility_sample["risk_score"].to_numpy()
)

alert_match = (
    new_predictions["alert"].to_numpy()
    == reproducibility_sample["alert"].to_numpy()
)

print(f"Observaciones comprobadas: {len(reproducibility_sample):,}")
print(f"Alertas coincidentes: {alert_match.mean():.2%}")
print(f"Diferencia máxima en risk_score: {risk_score_difference.max():.12f}")
print(f"Diferencia media en risk_score: {risk_score_difference.mean():.12f}")

Observaciones comprobadas: 10,000
Alertas coincidentes: 100.00%
Diferencia máxima en risk_score: 0.000000147452
Diferencia media en risk_score: 0.000000018614


#### Interpretación

La reconstrucción del sistema reprodujo correctamente las predicciones previamente persistidas para las 10,000 observaciones analizadas, obteniéndose una coincidencia del 100.00 % en las decisiones de alerta.

Las diferencias observadas en `risk_score` fueron únicamente del orden de 10⁻⁷, con una diferencia máxima de 0.000000147452 y media de 0.000000018614, magnitudes atribuibles a precisión numérica y sin efecto sobre las decisiones generadas.

Estos resultados confirman que el procedimiento predictivo puede reconstruirse a partir de los artefactos persistidos sin reentrenar el modelo y manteniendo las decisiones del sistema original.

## 4. Ejecución sobre nuevas observaciones

Una vez demostrada la reproducibilidad del procedimiento predictivo, el sistema puede aplicarse a nuevas observaciones siempre que estas proporcionen las 10 variables requeridas por el modelo.

La inferencia utiliza exclusivamente información compatible con el esquema definido durante el entrenamiento. El nuevo conjunto de datos debe proporcionar las variables categóricas y numéricas requeridas, mientras que `ARR_DEL15` no es necesario para generar una alerta, ya que representa el resultado que se pretende anticipar.

El procedimiento recuperado aplica automáticamente el preprocesamiento persistido, genera `risk_score` y utiliza el umbral definitivo de 0.406588 para producir `alert`.

La salida operativa del sistema queda formada por dos elementos principales:

- `risk_score`: puntuación continua utilizada para ordenar las observaciones según el riesgo estimado por el modelo;
- `alert`: decisión binaria que identifica las observaciones cuya puntuación alcanza el umbral establecido.

Por tanto, la aplicación del sistema sobre nuevos datos puede realizarse directamente mediante `predict_alerts()`, sin volver a ajustar transformadores, reentrenar el modelo o seleccionar un nuevo umbral.

In [17]:
# ---------------------------------------------------------
# 1. Simulación de un nuevo lote de observaciones
# ---------------------------------------------------------

new_observations = reproducibility_sample[
    input_features
].head(5).copy()


# ---------------------------------------------------------
# 2. Ejecución del sistema predictivo
# ---------------------------------------------------------

new_predictions = predict_alerts(
    new_observations
)


# ---------------------------------------------------------
# 3. Resultado operativo
# ---------------------------------------------------------

display(
    new_predictions[
        input_features + ["risk_score", "alert"]
    ]
)

,MONTH,DAY_OF_WEEK,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,ORIGIN,DEST,DEP_TIME_BLK,ARR_TIME_BLK,CRS_ELAPSED_TIME,DISTANCE,risk_score,alert
0,1,4,AA,AA,LAX,BOS,1300-1359,2200-2259,331.0,2611.0,0.607370,1
1,1,4,AA,AA,JFK,LAX,0900-0959,1300-1359,382.0,2475.0,0.470523,1
2,1,4,AA,AA,LAX,BOS,2200-2259,0600-0659,328.0,2611.0,0.509266,1
3,1,4,AA,AA,JFK,LAX,1700-1759,2100-2159,375.0,2475.0,0.669539,1
4,1,4,AA,AA,LAX,JFK,0800-0859,1600-1659,324.0,2475.0,0.492178,1


#### Interpretación

El procedimiento reconstruido puede aplicarse directamente a nuevos lotes de observaciones y generar para cada vuelo su correspondiente `risk_score` y decisión `alert`.

En las cinco observaciones utilizadas como demostración, las puntuaciones superaron el umbral de 0.406588 y fueron clasificadas como alerta. Esto confirma el funcionamiento operativo del flujo de inferencia, sin necesidad de reentrenar el modelo ni reajustar sus transformadores.

## 5. Persistencia de la configuración reproducible

La última etapa consiste en documentar los componentes necesarios para trasladar el procedimiento de inferencia a otro entorno de ejecución.

El sistema no requiere conservar los datasets utilizados durante el entrenamiento para generar nuevas predicciones. La inferencia depende del codificador categórico, el escalador numérico, el modelo final y la configuración que define las variables de entrada y el umbral de decisión.

Estos artefactos ya se encuentran persistidos individualmente en `results/modeling`. Por tanto, no es necesario generar copias adicionales de los modelos. Se preparará únicamente un manifiesto que documente los componentes necesarios y su función dentro del procedimiento de inferencia.

Este enfoque evita duplicar archivos y establece explícitamente las dependencias mínimas necesarias para reconstruir el sistema predictivo.

### 5.1 Construcción del manifiesto de inferencia

El manifiesto de inferencia permitirá identificar de forma explícita los artefactos que deben acompañar al sistema predictivo cuando se traslade a otro entorno.

Para cada componente se registrará su archivo, función y obligatoriedad. El manifiesto no sustituye a los artefactos originales, sino que actúa como documentación técnica de la configuración reproducible.

Se incluirán exclusivamente los cuatro productos necesarios para ejecutar nuevas predicciones, excluyendo resultados de GridSearch, probabilidades de validación, métricas y productos de evaluación que no intervienen durante la inferencia.

In [18]:
# ---------------------------------------------------------
# 1. Construcción del manifiesto de inferencia
# ---------------------------------------------------------

inference_manifest = pd.DataFrame(
    [
        {
            "component": "Codificador categórico",
            "file": encoder_path.name,
            "purpose": "Transformación de las 8 variables categóricas",
            "required": True,
        },
        {
            "component": "Escalador numérico",
            "file": scaler_path.name,
            "purpose": "Transformación de las 2 variables numéricas",
            "required": True,
        },
        {
            "component": "Modelo predictivo",
            "file": model_path.name,
            "purpose": "Generación del risk_score",
            "required": True,
        },
        {
            "component": "Configuración",
            "file": configuration_path.name,
            "purpose": "Variables, parámetros y umbral de decisión",
            "required": True,
        },
    ]
)


# ---------------------------------------------------------
# 2. Persistencia del manifiesto
# ---------------------------------------------------------

manifest_path = (
    modeling_path
    / "inference_manifest.csv"
)

inference_manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8",
)


# ---------------------------------------------------------
# 3. Resultado
# ---------------------------------------------------------

display(inference_manifest)

print(f"\nManifiesto persistido: {manifest_path}")

,component,file,purpose,required
0,Codificador categórico,final_categorical_encoder_2022_2025.joblib,Transformación de las 8 variables categóricas,True
1,Escalador numérico,final_numerical_scaler_2022_2025.joblib,Transformación de las 2 variables numéricas,True
2,Modelo predictivo,final_logistic_regression_2022_2025.joblib,Generación del risk_score,True
3,Configuración,final_model_configuration.json,"Variables, parámetros y umbral de decisión",True



Manifiesto persistido: G:\My Drive\MASTER Big Data\TFM\results\modeling\inference_manifest.csv


#### Interpretación

El manifiesto de inferencia quedó persistido correctamente e identifica los cuatro componentes necesarios para ejecutar el sistema predictivo: codificador categórico, escalador numérico, modelo final y configuración.

Con este producto queda documentada la dependencia mínima del sistema para inferencia reproducible, sin duplicar artefactos ni incorporar resultados de evaluación que no intervienen en la generación de nuevas predicciones.